## Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from csv import QUOTE_NONE

from recommenders.models.newsrec.newsrec_utils import prepare_hparams
from recommenders.models.newsrec.models.nrms import NRMSModel
from recommenders.models.newsrec.io.mind_iterator import MINDIterator

print("System version: {}".format(sys.version))
print("Tensorflow version: {}".format(tf.__version__))


2026-03-03 15:24:26.267576: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 15:24:26.269798: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-03 15:24:26.309162: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-03 15:24:26.309196: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-03 15:24:26.310424: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

System version: 3.11.14 (main, Feb 12 2026, 00:42:50) [Clang 21.1.4 ]
Tensorflow version: 2.15.1


In [2]:
epochs = 5
seed = 42
batch_size = 32

# Options: demo, small, large
MIND_type = 'small'

# Root folder containing the extracted MIND splits
LOCAL_DATA_ROOT = Path(os.getenv('MIND_DATA_ROOT', 'smallDataset')).expanduser()


## Load

In [3]:

LOCAL_DATA_ROOT = Path(os.environ.get("MIND_DATA_ROOT", "./smallDataset")).expanduser().resolve()

train_dir = LOCAL_DATA_ROOT / f"MINDsmall_train"
valid_dir = LOCAL_DATA_ROOT / f"MINDsmall_dev"

train_news_path      = str(train_dir / "news.tsv")
train_behaviors_path = str(train_dir / "behaviors.tsv")
valid_news_path      = str(valid_dir / "news.tsv")
valid_behaviors_path = str(valid_dir / "behaviors.tsv")

NEWS_COLUMNS = [
    "nid",
    "vertical",
    "subvertical",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

def load_tsv(path_str: str, columns):
    return pd.read_table(
        path_str,
        header=None,
        names=columns,
        sep='\t',
        quoting=QUOTE_NONE,
        dtype=object,
        na_filter=False,
    )

train_news_df = load_tsv(train_news_path, NEWS_COLUMNS)
valid_news_df = load_tsv(valid_news_path, NEWS_COLUMNS)
train_behaviors_df = load_tsv(train_behaviors_path, BEHAVIOR_COLUMNS)
valid_behaviors_df = load_tsv(valid_behaviors_path, BEHAVIOR_COLUMNS)

print(f"Loaded {len(train_news_df)} train news rows and {len(valid_news_df)} validation news rows from {LOCAL_DATA_ROOT}")
print(f"Loaded {len(train_behaviors_df)} train behaviors rows and {len(valid_behaviors_df)} validation behaviors rows")


Loaded 51282 train news rows and 42416 validation news rows from /home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset
Loaded 156965 train behaviors rows and 73152 validation behaviors rows


## Hyperparams

In [4]:
WORD_EMBEDDING_DIM = int(os.environ.get('WORD_EMBEDDING_DIM', '300'))
utils_dir = (LOCAL_DATA_ROOT / 'utils').resolve()
yaml_path = (LOCAL_DATA_ROOT / 'configs' / 'nrms.yaml').resolve()

def _must_exist(path: Path, label: str) -> str:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return str(path)

wordEmb_file = _must_exist(utils_dir / f"word_embeddings_{WORD_EMBEDDING_DIM}.npy", 'Word embeddings')
wordDict_file = _must_exist(utils_dir / 'word_dict.pkl', 'Word dictionary')
userDict_file = _must_exist(utils_dir / 'user_dict.pkl', 'User dictionary')
yaml_file = _must_exist(yaml_path, 'NRMS config')


In [5]:
hparams = prepare_hparams(yaml_file, 
                          wordEmb_file=wordEmb_file,
                          wordDict_file=wordDict_file, 
                          userDict_file=userDict_file,
                          batch_size=batch_size,
                          epochs=epochs,
                          show_step=10)
print(hparams)

HParams object with values {'support_quick_scoring': False, 'dropout': 0.2, 'attention_hidden_dim': 200, 'head_num': 15, 'head_dim': 16, 'filter_num': 200, 'window_size': 3, 'vert_emb_dim': 100, 'subvert_emb_dim': 100, 'gru_unit': 400, 'type': 'ini', 'user_emb_dim': 50, 'learning_rate': 0.0001, 'optimizer': 'adam', 'epochs': 5, 'batch_size': 32, 'show_step': 10, 'model_type': 'nrms', 'dataset': 'mind', 'data_format': 'news', 'npratio': 4, 'wordEmb_file': '/home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset/utils/word_embeddings_300.npy', 'wordDict_file': '/home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset/utils/word_dict.pkl', 'userDict_file': '/home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset/utils/user_dict.pkl', 'title_size': 30, 'word_emb_dim': 300, 'his_size': 50, 'loss': 'cross_entropy_loss', 'metrics': ['group_auc', 'mean_mrr', 'ndcg@5', 'ndcg@10']}


## Train model

Note this took approx. X mins last time 

In [6]:
nrms_model = NRMSModel(hparams, MINDIterator, seed=seed)

nrms_model.fit(
    train_news_file=train_news_path,
    train_behaviors_file=train_behaviors_path,
    valid_news_file=valid_news_path,
    valid_behaviors_file=valid_behaviors_path,
)

eval_metrics = nrms_model.run_eval(
    valid_news_file=valid_news_path,
    valid_behaviors_file=valid_behaviors_path,
)

for metric, value in eval_metrics.items():
    print(f"{metric}: {value:.6f}")


2026-03-03 15:24:40.792676: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-03-03 15:24:40.806617: W tensorflow/c/c_api.cc:305] Operation '{name:'embedding/embeddings/Assign' id:26 op device:{requested: '', assigned: ''} def:{{{node embedding/embeddings/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](embedding/embeddings, embedding/embeddings/Initializer/stateless_random_uniform)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
/home/asmvas/.venvs/recommenders/lib/python3.11/site-packages/keras/src/optimizers/legacy/adam.py:118: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)
0it [00:00, ?it/s]2026-03-03 15:24:51.484204: W tensorflow/c/c_api.cc:30

KeyboardInterrupt: 